# KingFisher Flex

```{device-card} thermo-fisher-kingfisher-flex
```

Run BindIt BDZ exports with a matching **96DW head, tip comb and plates**.
Tested with firmware 1.01.0 on macOS.

**Experimental:** command completion does not verify tip-comb attachment. Inspect
pickup and clearances.

See [installation](../installation.md) and [command-line tools](../cli.md).


## Physical setup

Prepare the 96DW head, matching tips and supported plates according to the instrument
instructions. Close BindIt before connecting. Run hardware cells individually.


## Inspect a protocol offline

Set the path to your BindIt export. Reading and preflight do not open USB.
`compatibility_issues()` lists blockers; `preflight()` raises if any remain.
`prompts()` shows plate actions and reagent additions. Plate labels are not tray numbers.


In [ ]:
from pathlib import Path
from pylabrobot.thermo_fisher.kingfisher.flex import KingFisherFlex, read_flex_bdz

protocol = read_flex_bdz(Path("protocol.bdz"))
commands = protocol.preflight()
(protocol.name, protocol.sha256, len(commands), protocol.prompts())

## Connect and identify

`setup()` opens USB and reads the identity without moving the instrument.
Pass `serial_number="..."` to select among multiple instruments.


In [ ]:
flex = KingFisherFlex()
await flex.setup()
(flex.version, flex.serial)

## Initialize

**This moves the instrument**, disables heating, and presents tray 4 at loading
with tray 1 at processing.


In [ ]:
await flex.initialize()
flex.processing_slot

## Observe events

Start one event consumer before running. Events include temperature readings,
plate requests, pauses, confirmations, failures, and run completion. Temperature
readings describe heater and ambient sensors, not well-liquid temperature.


In [ ]:
import asyncio
import logging

logging.basicConfig(level=logging.INFO)


async def observe_events():
  async for event in flex.events():
    logging.info("%s: %s; prompt=%s", event.kind, event.text, event.prompt)


observer = asyncio.create_task(observe_events())

## Confirm plate actions and pauses

The input thread lets telemetry continue while you respond. Return exactly `True`
after completing the action. Pass `confirm_prompt` for loading/unloading and
`confirm_pause` for protocol pauses. Rejection or callback failure invalidates the session.


In [ ]:
async def confirm_action(prompt):
  print(f"{prompt.action.upper()}: {prompt.label}")
  print(f"Loading tray {prompt.loading_slot}; processing tray {prompt.processing_slot}")
  answer = await asyncio.to_thread(
    input, "Complete the action, clear your hands, then type CONFIRM: "
  )
  return answer.strip() == "CONFIRM"

## Run a local protocol

This starts motion and may start heating. Complete each action before confirming
in Python. Reagent prompts include the plate, reagent, and volume per well.
`operator_timeout=None` allows an indefinite confirmation wait.

To use the instrument's physical Start button, omit both confirmation callbacks.
Callbacks replace the corresponding instrument pauses; they cannot advance an
already active onboard pause. Pause text mentioning seconds does not set a timer.

The call returns after execution and finalization. Heating follows the exported
commands; setting a temperature does not itself wait for target attainment.


In [ ]:
result = await flex.run_protocol(
  protocol,
  confirm_prompt=confirm_action,
  confirm_pause=confirm_action,
  operator_timeout=None,
)
result

## List and run onboard protocols

As an alternative to a local BDZ, list the protocols stored on the instrument:


In [ ]:
names = await flex.list_protocols()
names


Choose an exact name from the inventory. The following call starts motion and
waits for completion. Internal loading pauses require the physical Start button;
Python confirmation callbacks are unavailable for onboard runs. Protocol upload
and remote resume are unsupported. Onboard launch has been tested only through
the first loading prompt.


In [ ]:
await flex.start_stored_protocol("Your stored protocol")


## Abort an active run

Call `await flex.abort()` from another task while execution is pending. Keep the run
in an `asyncio.Task` when it needs a supervising task. Canceling a Python task or
interrupting a notebook cell does not stop the instrument.

After interruption, inspect physical positions, call `stop()`, and reconnect before
restarting. There is no automatic retry or resume.


## Disconnect

`stop()` closes USB and ends the event stream. It does not stop motion.


In [ ]:
await flex.stop()
await observer